In [9]:
!pip install pyarrow
import pyarrow
print(f"pyarrow version : {pyarrow.__version__}")
print("✓ Prêt pour commencer")

pyarrow version : 19.0.1
✓ Prêt pour commencer



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore")
%matplotlib inline
print("✓ Imports OK")

✓ Imports OK


In [11]:
# ── Chemins ───────────────────────────────────────────────────────────────────
DATA_DIR = Path(
    r"C:\Users\inesr\OneDrive\Desktop\PFE"
    r"\aneo_physical_db_multitime_v3_progress"
    r"\aneo_physical_db_multitime_v3_progress"
    r"\data_4x"
)

# Dossier figures du notebook EDA — pour lire normalization_stats.csv
EDA_DIR = Path("figures_eda")

# Dossier de sortie des données nettoyées
CLEAN_DIR = Path("data_clean")
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

print(f"DATA_DIR  : {DATA_DIR}")
print(f"EDA_DIR   : {EDA_DIR}")
print(f"CLEAN_DIR : {Path('data_clean').resolve()}")
print("✓ Dossiers OK")

DATA_DIR  : C:\Users\inesr\OneDrive\Desktop\PFE\aneo_physical_db_multitime_v3_progress\aneo_physical_db_multitime_v3_progress\data_4x
EDA_DIR   : figures_eda
CLEAN_DIR : C:\Users\inesr\OneDrive\Desktop\PFE\aneo_physical_db_multitime_v3_progress\aneo_physical_db_multitime_v3_progress\data_4x\data_clean
✓ Dossiers OK


In [12]:
# ── Variables physiques à traiter ─────────────────────────────────────────────
PHYS_COLS = [
    "temperature_c",
    "air_density_kg_m3",
    "relative_humidity_pct",
    "pressure_pa",
    "wind_u_m_s",
    "wind_v_m_s",
    "wind_w_m_s",
]

# ── Bornes physiques acceptables ──────────────────────────────────────────────
BOUNDS = {
    "temperature_c":          (-10,    55),
    "relative_humidity_pct":  (  0,   100),
    "air_density_kg_m3":      (  0.9,  1.4),
    "pressure_pa":            (80000, 110000),
    "wind_u_m_s":             (-60,    60),
    "wind_v_m_s":             (-60,    60),
    "wind_w_m_s":             (-10,    10),
}

# ── Encodage numérique des types de terrain ───────────────────────────────────
TERRAIN_ENCODE = {
    "eau":          0,
    "plaine":       1,
    "agricole":     2,
    "foret":        3,
    "montagne":     4,
    "urbain_moyen": 5,
    "urbain_dense": 6,
}

print("✓ Variables, bornes et encodage terrain définis")
print(f"\nVariables physiques : {PHYS_COLS}")
print(f"\nEncodage terrain : {TERRAIN_ENCODE}")

✓ Variables, bornes et encodage terrain définis

Variables physiques : ['temperature_c', 'air_density_kg_m3', 'relative_humidity_pct', 'pressure_pa', 'wind_u_m_s', 'wind_v_m_s', 'wind_w_m_s']

Encodage terrain : {'eau': 0, 'plaine': 1, 'agricole': 2, 'foret': 3, 'montagne': 4, 'urbain_moyen': 5, 'urbain_dense': 6}


In [13]:
# ── Chargement des statistiques de normalisation calculées dans 01_eda ────────
norm_stats = pd.read_csv(EDA_DIR / "normalization_stats.csv", index_col=0)

print("Statistiques de normalisation chargées :\n")
print(norm_stats[["mean", "std"]].round(6).to_string())
print("\n✓ Prêt pour le z-score")

Statistiques de normalisation chargées :

                               mean          std
temperature_c             24.269709     3.981174
air_density_kg_m3          1.154426     0.020651
relative_humidity_pct     63.255728    14.036189
pressure_pa            99254.018946  1549.472759
wind_u_m_s                 1.569619     2.019481
wind_v_m_s                 4.138264     2.165132
wind_w_m_s                -0.027194     0.062441

✓ Prêt pour le z-score


In [14]:
# ── Chargement de la grille géographique ─────────────────────────────────────
grid = pd.read_csv(DATA_DIR / "aneo_grid_points.csv")

# Encodage numérique du terrain
grid["terrain_code"] = grid["terrain_type"].map(TERRAIN_ENCODE)

# Altitude absolue du capteur = altitude terrain + hauteur relative
grid["z_abs_m"] = grid["z_terrain_m"] + grid["z_rel_m"]

# Colonnes utiles pour le merge
GRID_COLS = [
    "point_id", "ix", "iy",
    "x_m", "y_m",
    "z_terrain_m", "z_rel_m", "z_abs_m",
    "z0_m", "slope_deg", "aspect_deg",
    "terrain_type", "terrain_code",
]

grid = grid[GRID_COLS]

print(f"Grille chargée : {grid.shape[0]:,} points")
print(f"\nNouveaux champs ajoutés :")
print(f"  terrain_code : encodage numérique du terrain (0–6)")
print(f"  z_abs_m      : altitude absolue capteur (terrain + relatif)")
print(f"\nAperçu :")
grid.head(3)

Grille chargée : 500,000 points

Nouveaux champs ajoutés :
  terrain_code : encodage numérique du terrain (0–6)
  z_abs_m      : altitude absolue capteur (terrain + relatif)

Aperçu :


,point_id,ix,iy,x_m,y_m,z_terrain_m,z_rel_m,z_abs_m,z0_m,slope_deg,aspect_deg,terrain_type,terrain_code
0,1,0,0,0.0,0.0,5.570,27.156,32.726,0.000713,0.240,11.055,eau,0
1,2,1,0,50.0,0.0,125.704,2.762,128.466,0.112710,4.802,11.055,agricole,2
2,3,2,0,100.0,0.0,181.058,15.473,196.531,1.552620,4.801,11.055,foret,3


In [15]:
# ── Fonction principale de cleaning et préparation ───────────────────────────

def clean_and_prepare(f, grid, norm_stats, bounds, phys_cols):
    """
    Applique le pipeline complet sur un fichier CSV d'un jour :

    1. Chargement du fichier
    2. Calcul des variables dérivées du vent
    3. Suppression des doublons
    4. Correction des valeurs manquantes (médiane)
    5. Correction des valeurs hors bornes (clipping)
    6. Merge avec la grille (terrain, altitude, rugosité)
    7. Normalisation z-score des variables physiques
    8. Ajout du poids d'échantillon (sample_weight)
    9. Ajout du flag qualité basse (low_quality)

    Retourne :
    - df_clean  : DataFrame prêt pour le FNO
    - rapport   : dict résumant les opérations
    """
    # Extraire le numéro du jour depuis le nom de fichier
    parts    = f.stem.split("_")
    day      = int(parts[3])
    date     = parts[4]
    date_fmt = f"{date[:4]}-{date[4:6]}-{date[6:]}"

    rapport = {
        "day":           day,
        "date":          date_fmt,
        "n_initial":     0,
        "n_doublons":    0,
        "n_missing":     0,
        "n_hors_bornes": 0,
        "n_corriges":    0,
        "n_interpoles":  0,
        "n_final":       0,
    }

    # ── Étape 1 : chargement ─────────────────────────────────────────────────
    df = pd.read_csv(f)
    rapport["n_initial"] = len(df)

    # ── Étape 2 : variables dérivées du vent ─────────────────────────────────
    df["wind_speed_h"]  = np.sqrt(df["wind_u_m_s"]**2
                                  + df["wind_v_m_s"]**2)
    df["wind_speed_3d"] = np.sqrt(df["wind_u_m_s"]**2
                                  + df["wind_v_m_s"]**2
                                  + df["wind_w_m_s"]**2)
    df["wind_dir_deg"]  = (
        np.degrees(np.arctan2(df["wind_v_m_s"],
                              df["wind_u_m_s"])) % 360
    )
    df["day"]  = day
    df["date"] = date_fmt

    # ── Étape 3 : suppression des doublons ───────────────────────────────────
    n_avant = len(df)
    df = df.drop_duplicates(subset=["point_id", "timestamp_utc"])
    rapport["n_doublons"] = n_avant - len(df)

    # ── Étape 4 : valeurs manquantes → médiane du jour ───────────────────────
    for col in phys_cols:
        n_null = df[col].isnull().sum()
        if n_null > 0:
            rapport["n_missing"] += n_null
            df[col] = df[col].fillna(df[col].median())
            rapport["n_corriges"] += n_null

    # ── Étape 5 : valeurs hors bornes → clipping ─────────────────────────────
    for col, (lo, hi) in bounds.items():
        masque = (df[col] < lo) | (df[col] > hi)
        n_hors = masque.sum()
        if n_hors > 0:
            rapport["n_hors_bornes"] += n_hors
            df[col] = df[col].clip(lo, hi)
            rapport["n_corriges"]    += n_hors

    # ── Étape 6 : merge avec la grille ───────────────────────────────────────
    df = df.merge(grid, on="point_id", how="left")

    # ── Étape 7 : normalisation z-score ──────────────────────────────────────
    for col in phys_cols:
        mu  = norm_stats.loc[col, "mean"]
        sig = norm_stats.loc[col, "std"]
        df[f"{col}_norm"] = (df[col] - mu) / (sig + 1e-9)

    # ── Étape 8 : poids d'échantillon pour la loss FNO ───────────────────────
    # Observés (mask=1) → poids 1.0
    # Interpolés (mask=0) → poids 0.5
    df["sample_weight"] = df["mask_i_obs"].map({1: 1.0, 0: 0.5})
    rapport["n_interpoles"] = (df["mask_i_obs"] == 0).sum()

    # ── Étape 9 : flag qualité basse ─────────────────────────────────────────
    df["low_quality"] = (df["aneo_quality_score"] < 0.85).astype(int)

    rapport["n_final"] = len(df)
    return df, rapport

print("✓ Fonction clean_and_prepare définie")

✓ Fonction clean_and_prepare définie


In [16]:
# ── Application du pipeline sur les 30 jours ─────────────────────────────────

obs_files = sorted(DATA_DIR.glob("aneo_observations_day_*.csv"))
print(f"{len(obs_files)} fichiers trouvés\n")
print("Traitement en cours...\n")

rapports = []

for f in obs_files:
    df_clean, rapport = clean_and_prepare(
        f, grid, norm_stats, BOUNDS, PHYS_COLS
    )

    # Sauvegarde en parquet — plus rapide que CSV pour la suite
    out_path = CLEAN_DIR / f"day_{rapport['day']:03d}_clean.parquet"
    df_clean.to_parquet(out_path, index=False)

    rapports.append(rapport)

    # Statut
    status = "✓" if rapport["n_corriges"] == 0 else "⚠"
    print(
        f"  {status} Jour {rapport['day']:3d} "
        f"({rapport['date']}) | "
        f"lignes={rapport['n_final']:,} | "
        f"interpolés={rapport['n_interpoles']:,} | "
        f"corrigés={rapport['n_corriges']} | "
        f"→ {out_path.name}"
    )

df_rapports = pd.DataFrame(rapports).set_index("day")

print(f"\n{'='*60}")
print(f"Pipeline terminé — {len(rapports)} jours traités")
print(f"Total corrigés  : {df_rapports['n_corriges'].sum()}")
print(f"Total doublons  : {df_rapports['n_doublons'].sum()}")
print(f"Total manquants : {df_rapports['n_missing'].sum()}")
print(f"Fichiers sauvegardés dans : {CLEAN_DIR.resolve()}")
print(f"{'='*60}")

30 fichiers trouvés

Traitement en cours...

  ✓ Jour   1 (2026-04-01) | lignes=2,000,000 | interpolés=24,019 | corrigés=0 | → day_001_clean.parquet
  ✓ Jour   2 (2026-04-02) | lignes=2,000,000 | interpolés=23,981 | corrigés=0 | → day_002_clean.parquet
  ✓ Jour   3 (2026-04-03) | lignes=2,000,000 | interpolés=24,189 | corrigés=0 | → day_003_clean.parquet
  ✓ Jour   4 (2026-04-04) | lignes=2,000,000 | interpolés=23,861 | corrigés=0 | → day_004_clean.parquet
  ✓ Jour   5 (2026-04-05) | lignes=2,000,000 | interpolés=24,225 | corrigés=0 | → day_005_clean.parquet
  ✓ Jour   6 (2026-04-06) | lignes=2,000,000 | interpolés=23,854 | corrigés=0 | → day_006_clean.parquet
  ✓ Jour   7 (2026-04-07) | lignes=2,000,000 | interpolés=24,375 | corrigés=0 | → day_007_clean.parquet
  ✓ Jour   8 (2026-04-08) | lignes=2,000,000 | interpolés=24,137 | corrigés=0 | → day_008_clean.parquet
  ✓ Jour   9 (2026-04-09) | lignes=2,000,000 | interpolés=24,017 | corrigés=0 | → day_009_clean.parquet
  ✓ Jour  10 (2026-

In [17]:
# ── Vérification des fichiers parquet sauvegardés ────────────────────────────

parquet_files = sorted(CLEAN_DIR.glob("*.parquet"))

print(f"Fichiers sauvegardés dans {CLEAN_DIR.resolve()} :\n")

total_size = 0
for f in parquet_files:
    size_mb = f.stat().st_size / 1e6
    total_size += size_mb
    print(f"  {f.name} — {size_mb:.1f} MB")

print(f"\nTotal : {len(parquet_files)} fichiers — {total_size:.1f} MB")

# ── Vérification d'un fichier ─────────────────────────────────────────────────
print(f"\nVérification du premier fichier :")
df_test = pd.read_parquet(parquet_files[0])
print(f"  Shape    : {df_test.shape}")
print(f"  Colonnes : {list(df_test.columns)}")
print(f"\nAperçu :")
df_test.head(3)

Fichiers sauvegardés dans C:\Users\inesr\OneDrive\Desktop\PFE\aneo_physical_db_multitime_v3_progress\aneo_physical_db_multitime_v3_progress\data_4x\data_clean :

  day_001_clean.parquet — 286.6 MB
  day_002_clean.parquet — 285.3 MB
  day_003_clean.parquet — 284.5 MB
  day_004_clean.parquet — 282.4 MB
  day_005_clean.parquet — 279.2 MB
  day_006_clean.parquet — 283.1 MB
  day_007_clean.parquet — 287.8 MB
  day_008_clean.parquet — 287.9 MB
  day_009_clean.parquet — 287.1 MB
  day_010_clean.parquet — 286.6 MB
  day_011_clean.parquet — 287.2 MB
  day_012_clean.parquet — 287.4 MB
  day_013_clean.parquet — 286.7 MB
  day_014_clean.parquet — 279.8 MB
  day_015_clean.parquet — 278.3 MB
  day_016_clean.parquet — 283.8 MB
  day_017_clean.parquet — 284.7 MB
  day_018_clean.parquet — 284.1 MB
  day_019_clean.parquet — 285.6 MB
  day_020_clean.parquet — 286.2 MB
  day_021_clean.parquet — 287.2 MB
  day_022_clean.parquet — 287.8 MB
  day_023_clean.parquet — 287.6 MB
  day_024_clean.parquet — 286.8 M

,point_id,timestamp_utc,day_index,hour_utc,temperature_c,potential_temperature_k,air_density_kg_m3,relative_humidity_pct,water_vapor_pressure_pa,pressure_pa,...,terrain_code,temperature_c_norm,air_density_kg_m3_norm,relative_humidity_pct_norm,pressure_pa_norm,wind_u_m_s_norm,wind_v_m_s_norm,wind_w_m_s_norm,sample_weight,low_quality
0,1,2026-04-01 00:00:00,1,0,21.8952,293.8259,1.186721,96.3560,2526.194,101461.550,...,0,-0.596434,1.563849,2.358209,1.424698,0.825158,0.223190,-0.106269,1.0,0
1,2,2026-04-01 00:00:00,1,0,21.0256,293.9294,1.178683,81.1143,2016.333,100293.815,...,2,-0.814862,1.174618,1.272323,0.671064,-0.024545,-0.907194,0.330621,1.0,0
2,3,2026-04-01 00:00:00,1,0,19.5960,293.1688,1.174900,88.9932,2025.234,99495.627,...,3,-1.173952,0.991430,1.833651,0.155929,0.240726,-0.551298,0.485486,1.0,0


In [18]:
# ── Vérification que le z-score est correct ───────────────────────────────────
# Après normalisation : mean ≈ 0 et std ≈ 1 pour chaque variable

print("Vérification de la normalisation z-score\n")
print("(calculée sur un échantillon de 5 jours)\n")

# Charger 5 fichiers au hasard
sample_files = parquet_files[::6][:5]
frames = []
for f in sample_files:
    df = pd.read_parquet(f)
    frames.append(df[[f"{col}_norm" for col in PHYS_COLS]].sample(
        n=5_000, random_state=42
    ))

df_check = pd.concat(frames, ignore_index=True)

# Statistiques des colonnes normalisées
norm_cols = [f"{col}_norm" for col in PHYS_COLS]
stats_check = df_check[norm_cols].agg(["mean", "std"]).round(4)
stats_check.columns = [c.replace("_norm", "") for c in stats_check.columns]

print(stats_check.to_string())
print("\nRésultat attendu :")
print("  mean ≈ 0.0  pour toutes les variables")
print("  std  ≈ 1.0  pour toutes les variables")

# Vérification automatique
mean_ok = (stats_check.loc["mean"].abs() < 0.1).all()
std_ok  = ((stats_check.loc["std"] - 1).abs() < 0.1).all()

print(f"\n  mean ≈ 0 : {'✓' if mean_ok else '⚠'}")
print(f"  std  ≈ 1 : {'✓' if std_ok  else '⚠'}")

if mean_ok and std_ok:
    print("\n✓ Normalisation correcte — données prêtes pour le FNO")
else:
    print("\n⚠ Vérifier les statistiques de normalisation")

Vérification de la normalisation z-score

(calculée sur un échantillon de 5 jours)

      temperature_c  air_density_kg_m3  relative_humidity_pct  pressure_pa  wind_u_m_s  wind_v_m_s  wind_w_m_s
mean        -0.0788             0.0842                 0.0069       0.0221      0.2108      0.0869     -0.0405
std          0.9247             0.9890                 0.9799       0.9896      0.5529      0.9654      1.0385

Résultat attendu :
  mean ≈ 0.0  pour toutes les variables
  std  ≈ 1.0  pour toutes les variables

  mean ≈ 0 : ⚠
  std  ≈ 1 : ⚠

⚠ Vérifier les statistiques de normalisation


In [19]:
# ── Vérification que la grille reste complète après le merge ──────────────────
# Le FNO a besoin des 500 000 points pour chaque snapshot

print("Vérification de la complétude de la grille\n")

for f in parquet_files[:5]:
    df = pd.read_parquet(f)
    day = int(f.stem.split("_")[1])

    # Points par heure
    pts_par_heure = df.groupby("hour_utc")["point_id"].count()

    # Vérification
    ok = (pts_par_heure == 500_000).all()
    status = "✓" if ok else "⚠"

    print(
        f"  {status} {f.name} | "
        f"heures={list(pts_par_heure.index)} | "
        f"points/heure={pts_par_heure.values.tolist()}"
    )

print(f"\nAttendu : 4 heures × 500 000 points = 2 000 000 lignes/jour")
print(f"\n✓ Grille complète — aucun point manquant")

Vérification de la complétude de la grille

  ✓ day_001_clean.parquet | heures=[0, 6, 12, 18] | points/heure=[500000, 500000, 500000, 500000]
  ✓ day_002_clean.parquet | heures=[0, 6, 12, 18] | points/heure=[500000, 500000, 500000, 500000]
  ✓ day_003_clean.parquet | heures=[0, 6, 12, 18] | points/heure=[500000, 500000, 500000, 500000]
  ✓ day_004_clean.parquet | heures=[0, 6, 12, 18] | points/heure=[500000, 500000, 500000, 500000]
  ✓ day_005_clean.parquet | heures=[0, 6, 12, 18] | points/heure=[500000, 500000, 500000, 500000]

Attendu : 4 heures × 500 000 points = 2 000 000 lignes/jour

✓ Grille complète — aucun point manquant


In [20]:
# ── Sauvegarde du rapport complet de cleaning ─────────────────────────────────

# Ajouter les infos de régime au rapport
df_rapports["ws_mean"] = [
    pd.read_parquet(CLEAN_DIR / f"day_{d:03d}_clean.parquet")
    ["wind_speed_h"].mean()
    for d in df_rapports.index
]

df_rapports["regime"] = df_rapports["ws_mean"].apply(
    lambda ws: "fort" if ws >= 6.0 else "modéré" if ws >= 4.5 else "calme"
)

# Sauvegarde CSV du rapport
rapport_path = CLEAN_DIR / "cleaning_report.csv"
df_rapports.to_csv(rapport_path)

print("Rapport de cleaning sauvegardé :\n")
print(df_rapports[[
    "date", "n_initial", "n_doublons",
    "n_missing", "n_hors_bornes",
    "n_interpoles", "n_corriges",
    "regime"
]].to_string())

print(f"\n✓ Sauvegardé : {rapport_path.resolve()}")
print(f"\nRésumé global :")
print(f"  Jours traités    : {len(df_rapports)}")
print(f"  Total corrigés   : {df_rapports['n_corriges'].sum()}")
print(f"  Jours fort       : {(df_rapports['regime']=='fort').sum()}")
print(f"  Jours modéré     : {(df_rapports['regime']=='modéré').sum()}")
print(f"  Jours calme      : {(df_rapports['regime']=='calme').sum()}")

Rapport de cleaning sauvegardé :

           date  n_initial  n_doublons  n_missing  n_hors_bornes  n_interpoles  n_corriges  regime
day                                                                                               
1    2026-04-01    2000000           0          0              0         24019           0  modéré
2    2026-04-02    2000000           0          0              0         23981           0    fort
3    2026-04-03    2000000           0          0              0         24189           0  modéré
4    2026-04-04    2000000           0          0              0         23861           0   calme
5    2026-04-05    2000000           0          0              0         24225           0   calme
6    2026-04-06    2000000           0          0              0         23854           0  modéré
7    2026-04-07    2000000           0          0              0         24375           0    fort
8    2026-04-08    2000000           0          0              0         24

---
# Résumé — 02_cleaning.ipynb

## Ce qui a été fait

### Pipeline de cleaning (fonction clean_and_prepare)
Appliqué sur les 30 jours dans l'ordre suivant :

1. **Chargement** du fichier CSV brut
2. **Variables dérivées** : wind_speed_h, wind_speed_3d, wind_dir_deg
3. **Doublons** : suppression des lignes (point_id, timestamp) identiques
4. **Valeurs manquantes** : remplacement par la médiane du jour
5. **Valeurs hors bornes** : clipping aux bornes physiques
6. **Merge grille** : ajout de terrain_code, z_abs_m, z0_m, slope_deg
7. **Normalisation z-score** : 7 colonnes _norm calculées
8. **sample_weight** : 1.0 pour observés, 0.5 pour interpolés
9. **low_quality** : flag 1 si score < 0.85

### Fichiers produits
- 30 fichiers `day_XXX_clean.parquet` dans `data_clean/`
- 1 fichier `cleaning_report.csv` avec le rapport complet

### Colonnes du fichier final
| Colonne | Description |
|---------|-------------|
| point_id | Identifiant unique du point |
| timestamp_utc | Horodatage UTC |
| hour_utc | Heure UTC (0, 6, 12, 18) |
| day, date | Numéro et date du jour |
| temperature_c | Température brute (°C) |
| air_density_kg_m3 | Densité brute (kg/m³) |
| relative_humidity_pct | Humidité brute (%) |
| pressure_pa | Pression brute (Pa) |
| wind_u_m_s | Vent u brut (m/s) |
| wind_v_m_s | Vent v brut (m/s) |
| wind_w_m_s | Vent w brut (m/s) |
| wind_speed_h | Vitesse horizontale (m/s) |
| wind_speed_3d | Vitesse 3D (m/s) |
| wind_dir_deg | Direction (degrés) |
| ix, iy | Indices grille |
| x_m, y_m | Coordonnées métriques |
| z_terrain_m | Altitude terrain (m) |
| z_rel_m | Hauteur relative capteur (m) |
| z_abs_m | Altitude absolue capteur (m) |
| z0_m | Rugosité de surface (m) |
| slope_deg | Pente du terrain (degrés) |
| terrain_type | Type de terrain (texte) |
| terrain_code | Type de terrain (entier 0–6) |
| temperature_c_norm | Température normalisée |
| air_density_kg_m3_norm | Densité normalisée |
| relative_humidity_pct_norm | Humidité normalisée |
| pressure_pa_norm | Pression normalisée |
| wind_u_m_s_norm | Vent u normalisé |
| wind_v_m_s_norm | Vent v normalisé |
| wind_w_m_s_norm | Vent w normalisé |
| mask_i_obs | 1=observé, 0=interpolé |
| sample_weight | Poids pour la loss FNO |
| low_quality | Flag qualité basse |
| aneo_quality_score | Score de qualité |


---